# 01 — Exploratory Data Analysis
Full EDA of ASD_Combined_Enhanced_Final.csv.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.config import DATA_DIR, SAMPLE_DIR, AQ10_CLINICAL_THRESHOLD
from src.features.pipeline import engineer_features

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':110, 'font.size':11})


In [ ]:
# Load — use sample if full dataset not present
try:
    df = pd.read_csv(DATA_DIR / 'processed' / 'ASD_Combined_Enhanced_Final.csv')
    print(f"Full dataset loaded: {df.shape}")
except FileNotFoundError:
    df = pd.read_csv(SAMPLE_DIR / 'ASD_sample_public.csv')
    print(f"⚠️  Full dataset not found. Using public sample: {df.shape}")

df = engineer_features(df)
df.head()


In [ ]:
# ── 1. Class + Age distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14,5))
cc = df['Class'].map({0:'No ASD',1:'ASD Traits'}).value_counts()
axes[0].bar(cc.index, cc.values, color=['#4C9BE8','#E8704C'], edgecolor='white', width=0.5)
for i,(lbl,v) in enumerate(cc.items()):
    axes[0].text(i, v+5, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylim(0, max(cc.values)*1.25)

axes[1].hist(df[df['Class']==0]['Age'], bins=25, alpha=0.6, color='#4C9BE8', label='No ASD', edgecolor='white')
axes[1].hist(df[df['Class']==1]['Age'], bins=25, alpha=0.6, color='#E8704C', label='ASD Traits', edgecolor='white')
axes[1].set_title('Age Distribution by Class', fontweight='bold')
axes[1].set_xlabel('Age (years)'); axes[1].set_ylabel('Count'); axes[1].legend()
plt.suptitle('Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── 2. AQ-10 Score structure ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14,5))
bins = np.arange(-0.5, 11.5, 1)
axes[0].hist(df[df['Class']==0]['Total_Score'], bins=bins, alpha=0.7, color='#4C9BE8', label='No ASD', edgecolor='white')
axes[0].hist(df[df['Class']==1]['Total_Score'], bins=bins, alpha=0.7, color='#E8704C', label='ASD Traits', edgecolor='white')
axes[0].axvline(AQ10_CLINICAL_THRESHOLD+0.5, color='black', linestyle='--', lw=2, label=f'Decision boundary ({AQ10_CLINICAL_THRESHOLD}/{AQ10_CLINICAL_THRESHOLD+1})')
axes[0].set_title('AQ-10 Score Distribution — Note: Score≥7 = ASD by dataset construction', fontweight='bold')
axes[0].set_xlabel('AQ-10 Total Score'); axes[0].legend()

xtab = pd.crosstab(df['Total_Score'], df['Class'].map({0:'No ASD',1:'ASD Traits'}))
sns.heatmap(xtab, annot=True, fmt='d', cmap='Blues', ax=axes[1], linewidths=0.3, annot_kws={'size':8})
axes[1].set_title('Score × Class — 578 ambiguous rows at Score=6', fontweight='bold')
plt.suptitle('AQ-10 Score Structure (explains near-perfect AUC)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# ── 3. Item endorsement rates ─────────────────────────────────────────────
q_cols = [f'A{i}' for i in range(1,11)]
rates = pd.DataFrame({'ASD':df[df['Class']==1][q_cols].mean(), 'No ASD':df[df['Class']==0][q_cols].mean()})
rates.plot(kind='bar', color=['#E8704C','#4C9BE8'], edgecolor='white', figsize=(12,5))
plt.title('AQ-10 Item Endorsement Rate by Class', fontweight='bold')
plt.xlabel('AQ-10 Item'); plt.ylabel('Proportion'); plt.xticks(rotation=0); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── 4. Age Band breakdown ─────────────────────────────────────────────────
band_order = ['Child','Adolescent','Adult','Older_Adult']
bc = df.groupby(['Age_Band','Class']).size().unstack(fill_value=0).reindex(band_order)
bp = bc.div(bc.sum(axis=1), axis=0)*100
bp.columns = ['No ASD','ASD Traits']
bp.plot(kind='bar', stacked=True, color=['#4C9BE8','#E8704C'], edgecolor='white', figsize=(10,5))
plt.title('ASD Prevalence by Age Band (%)', fontweight='bold')
plt.xlabel('Age Band'); plt.ylabel('Percentage'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

print("\nKey statistics:")
print(f"  Overall ASD rate    : {df['Class'].mean()*100:.1f}%")
print(f"  ASD rate — Male     : {df[df['Sex']==1]['Class'].mean()*100:.1f}%")
print(f"  ASD rate — Female   : {df[df['Sex']==0]['Class'].mean()*100:.1f}%")
print(f"  Mean score (ASD=1)  : {df[df['Class']==1]['Total_Score'].mean():.2f}")
print(f"  Mean score (ASD=0)  : {df[df['Class']==0]['Total_Score'].mean():.2f}")
